In [1]:
%pip install "google-cloud-pipeline-components" "dotenv"


Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip freeze

absl-py @ file:///home/conda/feedstock_root/build_artifacts/absl-py_1783083173550/work
aiohappyeyeballs==2.7.1
aiohttp==3.14.3
aiohttp-cors==0.8.1
aiosignal==1.4.0
annotated-doc==0.0.5
annotated-types==0.8.0
anyio==4.15.1
argon2-cffi==25.1.0
argon2-cffi-bindings==26.1.0
array_record==0.8.3
arrow==1.4.0
asttokens==3.0.2
astunparse==1.6.3
async-lru==2.3.0
atpublic==5.1
attrs==26.1.0
babel==2.18.0
beautifulsoup4==4.15.0
bigframes==2.49.0
bleach==6.4.0
blessed==1.49.0
bq_stats @ file:///tmp/environments/base/pip/packages/bq_stats
cachetools==5.5.2
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.1
cheroot==11.1.2
click==8.5.0
click-option-group==0.5.7
cloud-tpu-client==0.7
cloud-tpu-profiler==2.4.0
cloudpickle==3.1.2
colorama==0.4.6
colorful==0.5.8
comm==0.2.3
contourpy==1.4.0
cryptography==42.0.8
cuda-bindings==13.4.1
cuda-pathfinder==1.8.1
cuda-toolkit==13.0.2
cupy-cuda13x==14.2.0
cycler==0.12.1
dataproc-spark-connect==1.1.0
db-dtypes==1.7.1
debugpy==1.8.21
decorator==5.3.1
defused

In [3]:
from kfp import dsl
from kfp.dsl import importer
from kfp import compiler

# v2 explicitly separates components into logical modules
from google_cloud_pipeline_components.v1.dataset import TabularDatasetCreateOp
from google_cloud_pipeline_components.v1.custom_job import CustomTrainingJobOp
from google_cloud_pipeline_components.v1.model import ModelUploadOp
from google_cloud_pipeline_components.v1.batch_predict_job import ModelBatchPredictOp
from google_cloud_pipeline_components.types import artifact_types
from kfp.dsl import importer_node
from kfp.dsl import Artifact

In [4]:
from pathlib import Path
from dotenv import load_dotenv
import os

# if running remotely, jpyter notebook won't allow direct upload of .env file.  
# Work around is: rename .env to env and then in the notebook, rename it back in terminal
env_path = Path.cwd() / ".env"
# env_path = Path(__file__).parent / "container" / ".env"
load_dotenv(dotenv_path=env_path)


True

In [6]:
PIPELINE_ROOT = f"gs://{os.environ['GCS_BUCKET']}/vertex-pipeline/pipeline_root"
PROJECT_ID = f"{os.environ['PROJECT_ID']}"

In [7]:
# 1. Define a component to split the BigQuery table
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-bigquery"]
)
def split_bq_dataset(
    project: str, 
    source_table: str, 
    train_table: str, 
    test_table: str
):
    from google.cloud import bigquery
    client = bigquery.Client(project=project)
    
    # Use BigQuery SQL to randomly assign rows to train or test tables
    # Example: 80% train, 20% test using FARM_FINGERPRINT
    train_query = f"""
        CREATE OR REPLACE TABLE `{train_table}` AS
        SELECT * FROM `{source_table}` as t
        WHERE MOD(ABS(FARM_FINGERPRINT(TO_JSON_STRING(t))), 100) < 80
    """
    client.query(train_query).result()
    
    test_query = f"""
        CREATE OR REPLACE TABLE `{test_table}` AS
        SELECT * FROM `{source_table}` as t
        WHERE MOD(ABS(FARM_FINGERPRINT(TO_JSON_STRING(t))), 100) >= 80
    """
    client.query(test_query).result()

In [8]:
@dsl.pipeline(
    name="beans-model-pipeline-v2",
    pipeline_root=PIPELINE_ROOT,
)
def pipeline(
    bq_source: str = f"{PROJECT_ID}.beans.large_dataset",
    display_name: str = "beans-model-pipeline",
    container_uri: str = f"us-central1-docker.pkg.dev/{PROJECT_ID}/cloud-run-source-deploy/{os.environ['DOCKER_IMAGE_NAME']}:{os.environ['DOCKER_IMAGE_TAG']}",
    project: str = PROJECT_ID,
    region: str = "us-central1",
):
    source = f"{PROJECT_ID}.beans.large_dataset"
    train_dest = f"{PROJECT_ID}.beans.train_data"
    test_dest = f"{PROJECT_ID}.beans.test_data"

    # 1. Create the Dataset
    # Run the split operation
    split_task = split_bq_dataset(
        project=project,
        source_table=source,
        train_table=train_dest,
        test_table=test_dest
    )
    
    model_output_dir = f"{PIPELINE_ROOT}/model"

    # 2. Run the Custom Training Container
    # In v2, you explicitly define the compute requirements
    training_op = CustomTrainingJobOp(
        display_name="pipeline-beans-custom-train",
        project=project,
        location=region,
        worker_pool_specs=[{
            "machine_spec": {
                "machine_type": "n1-standard-4",
            },
            "replica_count": 1,
            "container_spec": {
                "image_uri": container_uri,
                "env": [
                    # Force train.py to save the model to our known KFP path
                    {"name": "AIP_MODEL_DIR", "value": model_output_dir},
                    # Manually inject the dataset URI that v1 used to do automatically
                    {"name": "AIP_TRAINING_DATA_URI", "value": bq_source},
                    {"name": "AIP_TEST_DATA_URI", "value": bq_source},
                    # {"name": "AIP_STORAGE_BUCKET", "value": f"{PIPELINE_ROOT}"}
                    {"name": "AIP_STORAGE_BUCKET", "value": f"{os.environ['GCS_BUCKET']}"}
                ]
            }
        }],
        base_output_directory=PIPELINE_ROOT,
    ).after(split_task)

    # 3. Import the saved model back into the pipeline as an Artifact
    import_unmanaged_model_op = importer(
        artifact_uri=model_output_dir,
        artifact_class=artifact_types.UnmanagedContainerModel,
        metadata={
            "containerSpec": {
                # This defines the pre-built serving container used for predictions
                "imageUri": "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-6:latest"
            }
        }
    ).after(training_op)

    # 4. Upload the Model to Vertex Model Registry
    # (Because the custom job doesn't auto-upload in v2, we do it explicitly)
    model_upload_op = ModelUploadOp(
        project=project,
        location=region,
        display_name="beans-model-pipeline",
        # Pass the explicit Artifact from the importer, not the string!
        unmanaged_container_model=import_unmanaged_model_op.outputs["artifact"]
    )
    
    # Wait for training to finish before uploading
    model_upload_op.after(training_op)

    # 5. Run the Batch Prediction
    batch_predict_op = ModelBatchPredictOp(
        project=project,
        location=region,
        job_display_name="beans-batchpred",
        model=model_upload_op.outputs["model"],
        gcs_source_uris=[f"{os.environ['GCS_BUCKET']}/vertex-pipeline/batch-predict/batch-examples.csv"], 
        instances_format="csv",
        predictions_format="jsonl",
        gcs_destination_output_uri_prefix=PIPELINE_ROOT,
        machine_type="n1-standard-4", 
        starting_replica_count=1,
        max_replica_count=1
    )

In [9]:
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path="beans-model-pipeline.json"
)

from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location="us-central1")

job = aiplatform.PipelineJob(
    display_name="beans-pipeline-job",
    template_path="beans-model-pipeline.json",
    pipeline_root=PIPELINE_ROOT,
    enable_caching=True
)

job.submit()

Creating PipelineJob
PipelineJob created. Resource name: projects/869928330868/locations/us-central1/pipelineJobs/beans-model-pipeline-v2-20260921030617
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/869928330868/locations/us-central1/pipelineJobs/beans-model-pipeline-v2-20260921030617')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/beans-model-pipeline-v2-20260921030617?project=869928330868
